In [ ]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, differential_evolution
from dataclasses import dataclass
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

## 2024-2025 Federal Tax Brackets
We'll use the 2024 tax brackets (adjusted for inflation) as a baseline.

In [ ]:
# 2024 Federal Tax Brackets (Married Filing Jointly)
# Each tuple: (upper_limit, rate)
TAX_BRACKETS_MFJ_2024 = [
    (23200, 0.10),
    (94300, 0.12),
    (201050, 0.22),
    (383900, 0.24),
    (487450, 0.32),
    (731200, 0.35),
    (float('inf'), 0.37)
]

# 2024 Federal Tax Brackets (Single)
TAX_BRACKETS_SINGLE_2024 = [
    (11600, 0.10),
    (47150, 0.12),
    (100525, 0.22),
    (191950, 0.24),
    (243725, 0.32),
    (609350, 0.35),
    (float('inf'), 0.37)
]

# Standard Deductions for 2024
STANDARD_DEDUCTION_MFJ_2024 = 29200
STANDARD_DEDUCTION_SINGLE_2024 = 14600
STANDARD_DEDUCTION_65_PLUS_ADDITION = 1550  # Additional for 65+ (per person)

print("Tax brackets loaded for 2024")

In [ ]:
@dataclass
class TaxpayerProfile:
    """Profile containing taxpayer information for Roth conversion planning."""
    name: str
    age: int
    filing_status: str  # 'single' or 'mfj' (married filing jointly)
    traditional_ira_balance: float
    roth_ira_balance: float
    other_taxable_income: float  # Social Security, pensions, wages, etc.
    state_tax_rate: float  # State income tax rate (0 for no state tax)
    expected_return: float  # Expected annual investment return
    inflation_rate: float  # Expected inflation rate
    target_bracket: float  # Target marginal tax bracket to fill (e.g., 0.22)
    spouse_age: int = None  # For MFJ, spouse's age
    
    def get_standard_deduction(self, year: int = 0) -> float:
        """Calculate standard deduction based on filing status and age."""
        inflation_factor = (1 + self.inflation_rate) ** year
        
        if self.filing_status == 'mfj':
            base = STANDARD_DEDUCTION_MFJ_2024
            # Add extra for 65+ (both spouses if applicable)
            extra = 0
            if self.age + year >= 65:
                extra += STANDARD_DEDUCTION_65_PLUS_ADDITION
            if self.spouse_age and self.spouse_age + year >= 65:
                extra += STANDARD_DEDUCTION_65_PLUS_ADDITION
        else:
            base = STANDARD_DEDUCTION_SINGLE_2024
            extra = STANDARD_DEDUCTION_65_PLUS_ADDITION if self.age + year >= 65 else 0
        
        return (base + extra) * inflation_factor
    
    def get_tax_brackets(self, year: int = 0) -> List[Tuple[float, float]]:
        """Get inflation-adjusted tax brackets."""
        inflation_factor = (1 + self.inflation_rate) ** year
        base_brackets = TAX_BRACKETS_MFJ_2024 if self.filing_status == 'mfj' else TAX_BRACKETS_SINGLE_2024
        
        adjusted_brackets = []
        for limit, rate in base_brackets:
            if limit == float('inf'):
                adjusted_brackets.append((limit, rate))
            else:
                adjusted_brackets.append((limit * inflation_factor, rate))
        
        return adjusted_brackets


# Example taxpayer profile
example_profile = TaxpayerProfile(
    name="Example Taxpayer",
    age=62,
    filing_status='mfj',
    traditional_ira_balance=500000,
    roth_ira_balance=100000,
    other_taxable_income=60000,  # Social Security + pension
    state_tax_rate=0.05,  # 5% state tax
    expected_return=0.06,  # 6% annual return
    inflation_rate=0.025,  # 2.5% inflation
    target_bracket=0.22,  # Fill up to 22% bracket
    spouse_age=60
)

print(f"Example profile created: {example_profile.name}")
print(f"Traditional IRA: ${example_profile.traditional_ira_balance:,.0f}")
print(f"Roth IRA: ${example_profile.roth_ira_balance:,.0f}")

## Tax Calculation Functions

In [ ]:
def calculate_federal_tax(taxable_income: float, brackets: List[Tuple[float, float]]) -> float:
    """Calculate federal income tax based on taxable income and brackets."""
    if taxable_income <= 0:
        return 0
    
    tax = 0
    prev_limit = 0
    
    for limit, rate in brackets:
        if taxable_income <= prev_limit:
            break
        
        taxable_in_bracket = min(taxable_income, limit) - prev_limit
        if taxable_in_bracket > 0:
            tax += taxable_in_bracket * rate
        
        prev_limit = limit
    
    return tax


def get_marginal_rate(taxable_income: float, brackets: List[Tuple[float, float]]) -> float:
    """Get the marginal tax rate for a given taxable income."""
    prev_limit = 0
    for limit, rate in brackets:
        if taxable_income <= limit:
            return rate
        prev_limit = limit
    return brackets[-1][1]


def calculate_total_tax(
    profile: TaxpayerProfile,
    conversion_amount: float,
    year: int = 0
) -> Tuple[float, float, float]:
    """Calculate total tax (federal + state) for a given conversion amount.
    
    Returns: (federal_tax, state_tax, total_tax)
    """
    # Get inflation-adjusted values
    standard_deduction = profile.get_standard_deduction(year)
    brackets = profile.get_tax_brackets(year)
    
    # Calculate adjusted other income for inflation
    other_income = profile.other_taxable_income * ((1 + profile.inflation_rate) ** year)
    
    # Total gross income
    gross_income = other_income + conversion_amount
    
    # Taxable income after standard deduction
    taxable_income = max(0, gross_income - standard_deduction)
    
    # Calculate federal tax
    federal_tax = calculate_federal_tax(taxable_income, brackets)
    
    # Calculate state tax (simplified - flat rate on taxable income)
    state_tax = taxable_income * profile.state_tax_rate
    
    total_tax = federal_tax + state_tax
    
    return federal_tax, state_tax, total_tax


# Test the tax calculation
test_conversion = 50000
fed_tax, state_tax, total_tax = calculate_total_tax(example_profile, test_conversion, 0)
print(f"Test conversion of ${test_conversion:,.0f}:")
print(f"  Federal Tax: ${fed_tax:,.2f}")
print(f"  State Tax: ${state_tax:,.2f}")
print(f"  Total Tax: ${total_tax:,.2f}")
print(f"  Effective Rate: {total_tax/test_conversion*100:.1f}%")

## Optimal Conversion Amount Calculator
Find the optimal conversion amount to fill up to a target tax bracket.

In [ ]:
def find_bracket_top(
    profile: TaxpayerProfile,
    target_rate: float,
    year: int = 0
) -> float:
    """Find the income level at the top of the target bracket."""
    brackets = profile.get_tax_brackets(year)
    
    for limit, rate in brackets:
        if rate == target_rate:
            return limit
    
    # If target rate not found, return the limit just before the next higher rate
    prev_limit = 0
    for limit, rate in brackets:
        if rate > target_rate:
            return prev_limit
        prev_limit = limit
    
    return brackets[-2][0]  # Return second-to-last bracket limit


def calculate_optimal_conversion(
    profile: TaxpayerProfile,
    year: int = 0
) -> Dict:
    """Calculate the optimal Roth conversion amount to fill target bracket.
    
    Returns a dictionary with conversion details.
    """
    standard_deduction = profile.get_standard_deduction(year)
    brackets = profile.get_tax_brackets(year)
    
    # Inflation-adjusted other income
    other_income = profile.other_taxable_income * ((1 + profile.inflation_rate) ** year)
    
    # Current taxable income without conversion
    current_taxable = max(0, other_income - standard_deduction)
    current_marginal_rate = get_marginal_rate(current_taxable, brackets)
    
    # Find top of target bracket
    bracket_top = find_bracket_top(profile, profile.target_bracket, year)
    
    # Calculate optimal conversion to fill target bracket
    # Taxable income at bracket top
    target_taxable = bracket_top
    
    # Gross income needed to achieve target taxable income
    target_gross = target_taxable + standard_deduction
    
    # Optimal conversion amount
    optimal_conversion = max(0, target_gross - other_income)
    
    # Calculate taxes at optimal conversion
    fed_tax, state_tax, total_tax = calculate_total_tax(profile, optimal_conversion, year)
    
    return {
        'year': year,
        'calendar_year': 2024 + year,
        'other_income': other_income,
        'standard_deduction': standard_deduction,
        'current_taxable_income': current_taxable,
        'current_marginal_rate': current_marginal_rate,
        'target_bracket': profile.target_bracket,
        'bracket_top': bracket_top,
        'optimal_conversion': optimal_conversion,
        'federal_tax': fed_tax,
        'state_tax': state_tax,
        'total_tax': total_tax,
        'effective_tax_rate': total_tax / optimal_conversion if optimal_conversion > 0 else 0
    }


# Calculate optimal conversion for year 0
result = calculate_optimal_conversion(example_profile, 0)
print("Optimal Conversion Analysis for Year 1 (2024):")
print(f"  Other Income: ${result['other_income']:,.0f}")
print(f"  Standard Deduction: ${result['standard_deduction']:,.0f}")
print(f"  Current Marginal Rate: {result['current_marginal_rate']*100:.0f}%")
print(f"  Target Bracket: {result['target_bracket']*100:.0f}%")
print(f"  Optimal Conversion: ${result['optimal_conversion']:,.0f}")
print(f"  Total Tax on Conversion: ${result['total_tax']:,.0f}")
print(f"  Effective Rate: {result['effective_tax_rate']*100:.1f}%")

## 5-Year Roth Conversion Optimization

In [ ]:
def optimize_5_year_conversions(
    profile: TaxpayerProfile,
    years: int = 5
) -> pd.DataFrame:
    """Calculate optimal Roth conversions over multiple years.
    
    This tracks IRA balances and calculates year-by-year optimal conversions.
    """
    results = []
    
    # Track balances
    trad_balance = profile.traditional_ira_balance
    roth_balance = profile.roth_ira_balance
    total_taxes_paid = 0
    total_converted = 0
    
    for year in range(years):
        # Calculate optimal conversion for this year
        conversion_info = calculate_optimal_conversion(profile, year)
        
        # Cap conversion at available traditional IRA balance
        actual_conversion = min(conversion_info['optimal_conversion'], trad_balance)
        
        # Recalculate tax for actual conversion amount
        fed_tax, state_tax, total_tax = calculate_total_tax(profile, actual_conversion, year)
        
        # Update balances
        # Apply growth first (beginning of year)
        trad_balance *= (1 + profile.expected_return)
        roth_balance *= (1 + profile.expected_return)
        
        # Then do conversion (end of year)
        trad_balance -= actual_conversion
        roth_balance += actual_conversion
        
        total_taxes_paid += total_tax
        total_converted += actual_conversion
        
        results.append({
            'Year': year + 1,
            'Calendar_Year': 2024 + year,
            'Age': profile.age + year,
            'Other_Income': conversion_info['other_income'],
            'Conversion_Amount': actual_conversion,
            'Federal_Tax': fed_tax,
            'State_Tax': state_tax,
            'Total_Tax': total_tax,
            'Effective_Rate': (total_tax / actual_conversion * 100) if actual_conversion > 0 else 0,
            'Trad_IRA_Balance': trad_balance,
            'Roth_IRA_Balance': roth_balance,
            'Cumulative_Converted': total_converted,
            'Cumulative_Taxes': total_taxes_paid
        })
    
    df = pd.DataFrame(results)
    return df


# Run the 5-year optimization
conversion_plan = optimize_5_year_conversions(example_profile, years=5)
print("5-Year Roth Conversion Plan")
print("=" * 80)
conversion_plan

In [ ]:
# Summary statistics
print("\n" + "=" * 60)
print("5-YEAR ROTH CONVERSION SUMMARY")
print("=" * 60)
print(f"\nInitial Traditional IRA Balance: ${example_profile.traditional_ira_balance:,.0f}")
print(f"Initial Roth IRA Balance: ${example_profile.roth_ira_balance:,.0f}")
print(f"\nTotal Amount Converted: ${conversion_plan['Conversion_Amount'].sum():,.0f}")
print(f"Total Taxes Paid: ${conversion_plan['Total_Tax'].sum():,.0f}")
print(f"Average Effective Tax Rate: {conversion_plan['Effective_Rate'].mean():.1f}%")
print(f"\nFinal Traditional IRA Balance: ${conversion_plan['Trad_IRA_Balance'].iloc[-1]:,.0f}")
print(f"Final Roth IRA Balance: ${conversion_plan['Roth_IRA_Balance'].iloc[-1]:,.0f}")
print(f"Final Total IRA Balance: ${conversion_plan['Trad_IRA_Balance'].iloc[-1] + conversion_plan['Roth_IRA_Balance'].iloc[-1]:,.0f}")

## Visualization

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Conversion amounts by year
ax1 = axes[0, 0]
ax1.bar(conversion_plan['Calendar_Year'], conversion_plan['Conversion_Amount'], 
        color='steelblue', edgecolor='navy')
ax1.set_xlabel('Year')
ax1.set_ylabel('Conversion Amount ($)')
ax1.set_title('Annual Roth Conversion Amounts')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
for i, v in enumerate(conversion_plan['Conversion_Amount']):
    ax1.text(conversion_plan['Calendar_Year'].iloc[i], v + 2000, f'${v:,.0f}', 
             ha='center', va='bottom', fontsize=9)

# Plot 2: IRA Balance Progression
ax2 = axes[0, 1]
years = conversion_plan['Calendar_Year']
ax2.fill_between(years, 0, conversion_plan['Trad_IRA_Balance'], 
                 alpha=0.7, label='Traditional IRA', color='coral')
ax2.fill_between(years, conversion_plan['Trad_IRA_Balance'], 
                 conversion_plan['Trad_IRA_Balance'] + conversion_plan['Roth_IRA_Balance'],
                 alpha=0.7, label='Roth IRA', color='mediumseagreen')
ax2.set_xlabel('Year')
ax2.set_ylabel('Balance ($)')
ax2.set_title('IRA Balance Progression')
ax2.legend(loc='upper right')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# Plot 3: Taxes paid by year
ax3 = axes[1, 0]
width = 0.35
x = np.arange(len(years))
ax3.bar(x - width/2, conversion_plan['Federal_Tax'], width, label='Federal Tax', color='indianred')
ax3.bar(x + width/2, conversion_plan['State_Tax'], width, label='State Tax', color='goldenrod')
ax3.set_xlabel('Year')
ax3.set_ylabel('Tax Amount ($)')
ax3.set_title('Taxes Paid on Conversions')
ax3.set_xticks(x)
ax3.set_xticklabels(years)
ax3.legend()
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# Plot 4: Cumulative conversions and taxes
ax4 = axes[1, 1]
ax4.plot(years, conversion_plan['Cumulative_Converted'], 'o-', 
         label='Cumulative Converted', color='steelblue', linewidth=2, markersize=8)
ax4.plot(years, conversion_plan['Cumulative_Taxes'], 's-', 
         label='Cumulative Taxes', color='indianred', linewidth=2, markersize=8)
ax4.set_xlabel('Year')
ax4.set_ylabel('Amount ($)')
ax4.set_title('Cumulative Conversions vs Taxes Paid')
ax4.legend()
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('roth_conversion_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nChart saved as 'roth_conversion_analysis.png'")

## Advanced Optimization: Minimize Lifetime Taxes

This section uses optimization to find the best conversion amounts that minimize total lifetime taxes, considering future RMDs and growth.

In [ ]:
def calculate_rmd_factor(age: int) -> float:
    """Get the RMD distribution period based on age (Uniform Lifetime Table)."""
    # Simplified RMD factors - starts at age 73
    rmd_table = {
        73: 26.5, 74: 25.5, 75: 24.6, 76: 23.7, 77: 22.9,
        78: 22.0, 79: 21.1, 80: 20.2, 81: 19.4, 82: 18.5,
        83: 17.7, 84: 16.8, 85: 16.0, 86: 15.2, 87: 14.4,
        88: 13.7, 89: 12.9, 90: 12.2, 91: 11.5, 92: 10.8,
        93: 10.1, 94: 9.5, 95: 8.9, 96: 8.4, 97: 7.8,
        98: 7.3, 99: 6.8, 100: 6.4
    }
    if age < 73:
        return 0  # No RMD required
    if age > 100:
        return 6.4
    return rmd_table.get(age, 6.4)


def project_lifetime_taxes(
    profile: TaxpayerProfile,
    conversion_amounts: List[float],
    projection_years: int = 30
) -> Dict:
    """Project lifetime taxes with given conversion strategy.
    
    Args:
        profile: Taxpayer profile
        conversion_amounts: List of conversion amounts for each year
        projection_years: Years to project (default 30)
    
    Returns:
        Dictionary with projection results
    """
    trad_balance = profile.traditional_ira_balance
    roth_balance = profile.roth_ira_balance
    
    total_conversion_taxes = 0
    total_rmd_taxes = 0
    yearly_data = []
    
    conversion_years = len(conversion_amounts)
    
    for year in range(projection_years):
        age = profile.age + year
        
        # Apply growth
        trad_balance *= (1 + profile.expected_return)
        roth_balance *= (1 + profile.expected_return)
        
        # Determine conversion (only during conversion period)
        if year < conversion_years:
            conversion = min(conversion_amounts[year], max(0, trad_balance))
        else:
            conversion = 0
        
        # Calculate RMD (if applicable)
        rmd_factor = calculate_rmd_factor(age)
        if rmd_factor > 0:
            rmd = trad_balance / rmd_factor
        else:
            rmd = 0
        
        # Total taxable distribution
        total_distribution = conversion + rmd
        
        # Calculate taxes
        _, _, tax = calculate_total_tax(profile, total_distribution, year)
        
        # Allocate taxes between conversion and RMD
        if total_distribution > 0:
            conversion_tax = tax * (conversion / total_distribution)
            rmd_tax = tax * (rmd / total_distribution)
        else:
            conversion_tax = 0
            rmd_tax = 0
        
        total_conversion_taxes += conversion_tax
        total_rmd_taxes += rmd_tax
        
        # Update balances
        trad_balance -= (conversion + rmd)
        roth_balance += conversion  # RMD is withdrawn, not converted
        
        yearly_data.append({
            'year': year,
            'age': age,
            'trad_balance': trad_balance,
            'roth_balance': roth_balance,
            'conversion': conversion,
            'rmd': rmd,
            'tax': tax
        })
    
    return {
        'total_conversion_taxes': total_conversion_taxes,
        'total_rmd_taxes': total_rmd_taxes,
        'total_lifetime_taxes': total_conversion_taxes + total_rmd_taxes,
        'final_trad_balance': trad_balance,
        'final_roth_balance': roth_balance,
        'yearly_data': yearly_data
    }


# Test with current strategy
current_conversions = conversion_plan['Conversion_Amount'].tolist()
projection = project_lifetime_taxes(example_profile, current_conversions, 30)

print("30-Year Projection with Current Strategy:")
print(f"  Total Conversion Taxes: ${projection['total_conversion_taxes']:,.0f}")
print(f"  Total RMD Taxes: ${projection['total_rmd_taxes']:,.0f}")
print(f"  Total Lifetime Taxes: ${projection['total_lifetime_taxes']:,.0f}")

In [ ]:
def optimize_conversions_scipy(
    profile: TaxpayerProfile,
    years: int = 5,
    projection_years: int = 30
) -> Tuple[List[float], Dict]:
    """Use scipy optimization to find optimal conversion amounts."""
    
    def objective(conversions):
        """Objective function to minimize total lifetime taxes."""
        result = project_lifetime_taxes(profile, list(conversions), projection_years)
        return result['total_lifetime_taxes']
    
    # Initial guess: equal conversions over the period
    initial_guess = [profile.traditional_ira_balance / years] * years
    
    # Bounds: 0 to full balance for each year
    bounds = [(0, profile.traditional_ira_balance) for _ in range(years)]
    
    # Use differential evolution for global optimization
    result = differential_evolution(
        objective,
        bounds,
        seed=42,
        maxiter=200,
        tol=100,
        workers=-1,
        updating='deferred'
    )
    
    optimal_conversions = list(result.x)
    optimal_projection = project_lifetime_taxes(profile, optimal_conversions, projection_years)
    
    return optimal_conversions, optimal_projection


print("Running optimization (this may take a moment)...")
optimal_conversions, optimal_result = optimize_conversions_scipy(example_profile, years=5)

print("\nOptimized 5-Year Conversion Strategy:")
print("=" * 50)
for i, conv in enumerate(optimal_conversions):
    print(f"  Year {i+1} ({2024+i}): ${conv:,.0f}")

print(f"\nTotal Converted: ${sum(optimal_conversions):,.0f}")
print(f"Lifetime Taxes (Optimized): ${optimal_result['total_lifetime_taxes']:,.0f}")
print(f"Lifetime Taxes (Bracket-Fill): ${projection['total_lifetime_taxes']:,.0f}")
print(f"Tax Savings: ${projection['total_lifetime_taxes'] - optimal_result['total_lifetime_taxes']:,.0f}")

## Comparison: Different Strategies

In [ ]:
def compare_strategies(profile: TaxpayerProfile) -> pd.DataFrame:
    """Compare different Roth conversion strategies."""
    strategies = {}
    
    # Strategy 1: No conversions
    no_conversion = project_lifetime_taxes(profile, [0, 0, 0, 0, 0], 30)
    strategies['No Conversion'] = no_conversion
    
    # Strategy 2: Fill 12% bracket
    profile_12 = TaxpayerProfile(**{**profile.__dict__, 'target_bracket': 0.12})
    plan_12 = optimize_5_year_conversions(profile_12, 5)
    result_12 = project_lifetime_taxes(profile, plan_12['Conversion_Amount'].tolist(), 30)
    strategies['Fill 12% Bracket'] = result_12
    
    # Strategy 3: Fill 22% bracket (current)
    result_22 = project_lifetime_taxes(profile, current_conversions, 30)
    strategies['Fill 22% Bracket'] = result_22
    
    # Strategy 4: Fill 24% bracket
    profile_24 = TaxpayerProfile(**{**profile.__dict__, 'target_bracket': 0.24})
    plan_24 = optimize_5_year_conversions(profile_24, 5)
    result_24 = project_lifetime_taxes(profile, plan_24['Conversion_Amount'].tolist(), 30)
    strategies['Fill 24% Bracket'] = result_24
    
    # Strategy 5: Optimized (from scipy)
    strategies['Optimized'] = optimal_result
    
    # Create comparison dataframe
    comparison_data = []
    for name, result in strategies.items():
        comparison_data.append({
            'Strategy': name,
            'Conversion_Taxes': result['total_conversion_taxes'],
            'RMD_Taxes': result['total_rmd_taxes'],
            'Total_Lifetime_Taxes': result['total_lifetime_taxes'],
            'Final_Roth_Balance': result['final_roth_balance'],
            'Final_Trad_Balance': result['final_trad_balance']
        })
    
    df = pd.DataFrame(comparison_data)
    df['Tax_Savings_vs_NoConversion'] = df['Total_Lifetime_Taxes'].iloc[0] - df['Total_Lifetime_Taxes']
    
    return df


comparison_df = compare_strategies(example_profile)
print("Strategy Comparison (30-Year Projection)")
print("=" * 80)
comparison_df

In [ ]:
# Visualize strategy comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Total lifetime taxes by strategy
ax1 = axes[0]
colors = ['lightcoral', 'lightsalmon', 'khaki', 'lightgreen', 'mediumseagreen']
bars = ax1.barh(comparison_df['Strategy'], comparison_df['Total_Lifetime_Taxes'], color=colors)
ax1.set_xlabel('Total Lifetime Taxes ($)')
ax1.set_title('30-Year Lifetime Tax Comparison')
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

# Add value labels
for bar, val in zip(bars, comparison_df['Total_Lifetime_Taxes']):
    ax1.text(val + 2000, bar.get_y() + bar.get_height()/2, 
             f'${val:,.0f}', va='center', fontsize=9)

# Plot 2: Final account balances
ax2 = axes[1]
x = np.arange(len(comparison_df))
width = 0.35
ax2.barh(x - width/2, comparison_df['Final_Trad_Balance'], width, 
         label='Traditional IRA', color='coral')
ax2.barh(x + width/2, comparison_df['Final_Roth_Balance'], width, 
         label='Roth IRA', color='mediumseagreen')
ax2.set_yticks(x)
ax2.set_yticklabels(comparison_df['Strategy'])
ax2.set_xlabel('Final Balance ($)')
ax2.set_title('Final IRA Balances After 30 Years')
ax2.legend()
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('strategy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nChart saved as 'strategy_comparison.png'")

## Interactive Configuration

Modify the profile below to analyze your own situation:

In [ ]:
# =============================================================================
# MODIFY YOUR PROFILE HERE
# =============================================================================

my_profile = TaxpayerProfile(
    name="Your Name",
    age=60,                           # Your current age
    filing_status='mfj',              # 'mfj' (married filing jointly) or 'single'
    traditional_ira_balance=400000,   # Your Traditional IRA balance
    roth_ira_balance=50000,           # Your Roth IRA balance
    other_taxable_income=50000,       # Other income (Social Security, pension, wages)
    state_tax_rate=0.05,              # Your state tax rate (0 if no state tax)
    expected_return=0.06,             # Expected investment return (6% = 0.06)
    inflation_rate=0.025,             # Expected inflation (2.5% = 0.025)
    target_bracket=0.22,              # Target tax bracket to fill
    spouse_age=58                     # Spouse age (if married)
)

# Run analysis
print(f"Analyzing profile for: {my_profile.name}")
print("=" * 60)

my_plan = optimize_5_year_conversions(my_profile, years=5)
print("\n5-Year Conversion Plan:")
display(my_plan[['Calendar_Year', 'Age', 'Conversion_Amount', 'Total_Tax', 'Effective_Rate', 
                 'Trad_IRA_Balance', 'Roth_IRA_Balance']])

print(f"\nTotal to Convert: ${my_plan['Conversion_Amount'].sum():,.0f}")
print(f"Total Taxes: ${my_plan['Total_Tax'].sum():,.0f}")
print(f"Average Effective Rate: {my_plan['Effective_Rate'].mean():.1f}%")

## Key Considerations & Disclaimers

### Important Notes:
1. **This is for educational purposes only** - Consult a qualified tax professional before making any conversion decisions
2. **Tax laws change** - Brackets and rules may be different in future years
3. **IRMAA implications** - Large conversions may increase Medicare premiums
4. **State taxes vary** - Some states don't tax retirement income
5. **Pro-rata rule** - If you have non-deductible Traditional IRA contributions, calculations will differ
6. **Capital gains** - Consider impact on investment income taxation
7. **Estate planning** - Roth conversions can be beneficial for heirs

### Factors Not Included in This Model:
- Medicare IRMAA surcharges
- Social Security taxation thresholds
- Net Investment Income Tax (NIIT)
- State-specific retirement income exclusions
- ACA subsidy impacts
- Estate tax considerations